## 1. Treinamento do Modelo Preditivo e Geração de Predições de UHI
Nesta etapa final, desenvolvemos e treinamos um modelo de Random Forest utilizando as variáveis derivadas de sensoriamento remoto, avaliamos sua capacidade preditiva e geramos estimativas do Índice de Ilha de Calor Urbana para os pontos de submissão:


In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor

In [2]:
# Carregar dados de treinamento com features extraídas
df_train = pd.read_csv('dataset/training_with_satellite_features.csv')

# Tratamento de valores ausentes - preenchimento com a mediana
for col in df_train.columns:
    if df_train[col].dtype != 'object' and col != 'id':
        df_train[col] = df_train[col].fillna(df_train[col].median())

# Seleção de features, excluindo coordenadas geográficas e identificadores
X_train = df_train.drop(['id', 'UHI Index', 'Latitude', 'Longitude'], axis=1)
y_train = df_train['UHI Index']

print("Features utilizadas no modelo:")
print(X_train.columns.tolist())

# Treinamento do modelo Random Forest
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Carregar features extraídas para os pontos de submissão
submission_features = pd.read_csv('dataset/submission_features.csv')

# Verificação de consistência entre features de treinamento e submissão
missing_cols = set(X_train.columns) - set(submission_features.columns)
extra_cols = set(submission_features.columns) - set(X_train.columns) - set(['Latitude', 'Longitude'])

if missing_cols:
    print(f"Atenção: Colunas ausentes nas features de submissão: {missing_cols}")
    # Adicionar colunas ausentes com valores nulos
    for col in missing_cols:
        submission_features[col] = np.nan

# Tratamento de valores ausentes nas features de submissão
for col in X_train.columns:
    if col in submission_features.columns and submission_features[col].isnull().any():
        median_val = df_train[col].median()
        print(f"Preenchendo {submission_features[col].isnull().sum()} valores nulos em '{col}' com mediana: {median_val}")
        submission_features[col] = submission_features[col].fillna(median_val)

# Seleção das mesmas features usadas no treinamento
X_submission = submission_features[X_train.columns]

# Geração de predições
predictions = model.predict(X_submission)

Features utilizadas no modelo:
['S2_B01', 'S2_B02', 'S2_B03', 'S2_B04', 'S2_B05', 'S2_B06', 'S2_B07', 'S2_B08', 'S2_B8A', 'S2_B11', 'S2_B12', 'NDVI_S2', 'NDBI_S2', 'NDWI_S2', 'NDVI_LS', 'LST_LS']


In [3]:
# Integração das predições ao template de submissão
submission_df = pd.read_csv('dataset/submission_template.csv')
submission_df['UHI Index'] = predictions

# Visualização das predições
print("\nPrevisões para os pontos de submissão:")
print(submission_df.head())

# Estatísticas básicas das predições
print("\nEstatísticas das predições:")
print(submission_df['UHI Index'].describe())

# Salvamento do arquivo final de submissão
submission_df.to_csv('submission_final.csv', index=False)
print("\nArquivo de submissão com predições salvo como 'submission_final.csv'")


Previsões para os pontos de submissão:
   Longitude   Latitude  UHI Index
0 -73.971665  40.788763   0.969054
1 -73.971928  40.788875   0.975914
2 -73.967080  40.789080   0.982860
3 -73.972550  40.789082   0.985043
4 -73.969697  40.787953   0.962425

Estatísticas das predições:
count    1040.000000
mean        0.999872
std         0.012036
min         0.961609
25%         0.992634
50%         1.000643
75%         1.008104
max         1.033564
Name: UHI Index, dtype: float64

Arquivo de submissão com predições salvo como 'submission_final.csv'
